# 01 - Explore Pretrained BioBERT for NER

This notebook explores the pretrained BioBERT model and establishes baseline performance on medical NER tasks.

**Goals:**
1. Load BioBERT and examine its architecture
2. Test zero-shot NER on sample medical text
3. Load BC5CDR dataset and compute baseline metrics
4. Understand the data format for fine-tuning

In [ ]:
# Install dependencies if needed (uncomment to run)
# !pip install transformers datasets torch accelerate seqeval

In [ ]:
import os
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification
from transformers import pipeline
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load and Examine BioBERT

In [ ]:
# Model name - using BioBERT base cased
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from {MODEL_NAME}...")
model = AutoModel.from_pretrained(MODEL_NAME)

print(f"\nModel architecture:")
print(f"  - Hidden size: {model.config.hidden_size}")
print(f"  - Num layers: {model.config.num_hidden_layers}")
print(f"  - Num attention heads: {model.config.num_attention_heads}")
print(f"  - Vocab size: {model.config.vocab_size}")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test tokenization on medical text
sample_text = "Patient presents with persistent headache and fever for 3 days."

tokens = tokenizer.tokenize(sample_text)
print(f"Original text: {sample_text}")
print(f"Tokens ({len(tokens)}): {tokens}")

# Show token to ID mapping
encoded = tokenizer(sample_text, return_tensors="pt")
print(f"\nToken IDs: {encoded['input_ids'][0].tolist()}")
print(f"Attention mask: {encoded['attention_mask'][0].tolist()}")

## 2. Test Zero-Shot NER with Existing NER Model

Let's see how a pretrained biomedical NER model performs (this gives us a baseline to compare against).

In [ ]:
# Try a pretrained biomedical NER model for baseline
# This model was fine-tuned on biomedical data
try:
    ner_pipeline = pipeline(
        "ner",
        model="d4data/biomedical-ner-all",
        aggregation_strategy="simple"
    )
    
    test_sentences = [
        "Patient has persistent headache and fever.",
        "Aspirin is commonly used to treat inflammation and pain.",
        "The patient was diagnosed with diabetes mellitus type 2.",
        "I have been experiencing chest pain and shortness of breath.",
    ]
    
    print("Zero-shot NER results from pretrained biomedical model:\n")
    for sentence in test_sentences:
        print(f"Text: {sentence}")
        entities = ner_pipeline(sentence)
        for ent in entities:
            print(f"  - {ent['word']}: {ent['entity_group']} (score: {ent['score']:.3f})")
        print()
        
except Exception as e:
    print(f"Could not load pretrained NER model: {e}")
    print("This is expected if the model is not available. We'll train our own!")

## 3. Load and Explore BC5CDR Dataset

In [ ]:
# Load BC5CDR dataset
print("Loading BC5CDR dataset...")
try:
    dataset = load_dataset("bigbio/bc5cdr", "bc5cdr_bigbio_kb", trust_remote_code=True)
    print(f"Dataset loaded successfully!")
    print(f"Splits: {list(dataset.keys())}")
    print(f"Train size: {len(dataset['train'])}")
    print(f"Validation size: {len(dataset['validation'])}")
    print(f"Test size: {len(dataset['test'])}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("You may need to install additional dependencies or check internet connection.")

In [ ]:
# Examine a single example
if 'dataset' in dir():
    example = dataset['train'][0]
    print("Example structure:")
    print(f"Keys: {example.keys()}")
    print(f"\nID: {example.get('id', 'N/A')}")
    print(f"\nPassages: {len(example.get('passages', []))}")
    if example.get('passages'):
        for i, passage in enumerate(example['passages'][:2]):
            print(f"  Passage {i}: {passage.get('text', [''])[0][:200]}...")
    print(f"\nEntities: {len(example.get('entities', []))}")
    if example.get('entities'):
        for entity in example['entities'][:5]:
            print(f"  - {entity.get('text', [''])} ({entity.get('type', 'unknown')})")

In [ ]:
# Count entity types in the dataset
if 'dataset' in dir():
    entity_counts = {"Disease": 0, "Chemical": 0, "Other": 0}
    
    for example in dataset['train']:
        for entity in example.get('entities', []):
            entity_type = entity.get('type', 'Other')
            if entity_type.lower() in ['disease', 'disorder']:
                entity_counts['Disease'] += 1
            elif entity_type.lower() in ['chemical', 'drug']:
                entity_counts['Chemical'] += 1
            else:
                entity_counts['Other'] += 1
    
    print("Entity distribution in training set:")
    for entity_type, count in entity_counts.items():
        print(f"  {entity_type}: {count:,}")
    
    # Visualize
    plt.figure(figsize=(8, 5))
    plt.bar(entity_counts.keys(), entity_counts.values(), color=['#e74c3c', '#3498db', '#95a5a6'])
    plt.title('Entity Distribution in BC5CDR Training Set')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 4. Understand BIO Tagging Format

NER uses BIO (Beginning-Inside-Outside) tagging:
- **B-Entity**: First token of an entity
- **I-Entity**: Continuation tokens of an entity
- **O**: Tokens not part of any entity

In [ ]:
# Demonstrate BIO tagging
demo_text = "Aspirin treats headache and fever."
demo_tokens = ["Aspirin", "treats", "headache", "and", "fever", "."]
demo_labels = ["B-Chemical", "O", "B-Disease", "O", "B-Disease", "O"]

print("BIO Tagging Example:")
print(f"Text: {demo_text}")
print("\n{:<12} {:<12}".format("Token", "Label"))
print("-" * 24)
for token, label in zip(demo_tokens, demo_labels):
    print(f"{token:<12} {label:<12}")

In [ ]:
# Multi-word entity example
demo_text2 = "Patient has diabetes mellitus type 2."
demo_tokens2 = ["Patient", "has", "diabetes", "mellitus", "type", "2", "."]
demo_labels2 = ["O", "O", "B-Disease", "I-Disease", "I-Disease", "I-Disease", "O"]

print("\nMulti-word Entity Example:")
print(f"Text: {demo_text2}")
print("\n{:<12} {:<12}".format("Token", "Label"))
print("-" * 24)
for token, label in zip(demo_tokens2, demo_labels2):
    print(f"{token:<12} {label:<12}")

## 5. Label Scheme for Our Model

In [ ]:
# Our label scheme
LABEL_LIST = [
    "O",           # 0 - Outside any entity
    "B-Disease",   # 1 - Beginning of disease entity
    "I-Disease",   # 2 - Inside disease entity  
    "B-Chemical",  # 3 - Beginning of chemical entity
    "I-Chemical",  # 4 - Inside chemical entity
    "B-Symptom",   # 5 - Beginning of symptom entity
    "I-Symptom",   # 6 - Inside symptom entity
]

LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {idx: label for idx, label in enumerate(LABEL_LIST)}

print("Label Mapping:")
for label, idx in LABEL_TO_ID.items():
    print(f"  {idx}: {label}")

print(f"\nTotal labels: {len(LABEL_LIST)}")

## 6. Test Data Processing Pipeline

In [ ]:
# Import our data preparation module
try:
    from data.prepare_data import process_bc5cdr_example, LABEL_LIST, LABEL_TO_ID
    
    if 'dataset' in dir():
        # Process an example
        example = dataset['train'][0]
        processed = process_bc5cdr_example(example)
        
        print("Processed example:")
        print(f"ID: {processed['id']}")
        print(f"Tokens ({len(processed['tokens'])}): {processed['tokens'][:20]}...")
        print(f"Labels ({len(processed['labels'])}): {processed['labels'][:20]}...")
        print(f"NER tags: {processed['ner_tags'][:20]}...")
        
        # Show token-label pairs for entities
        print("\nEntities found:")
        for i, (token, tag) in enumerate(zip(processed['tokens'], processed['ner_tags'])):
            if tag != 'O':
                print(f"  {token}: {tag}")
                
except ImportError as e:
    print(f"Could not import data preparation module: {e}")
    print("Run the prepare_data.py script first or check the import path.")

## 7. Subword Tokenization and Label Alignment

BioBERT uses WordPiece tokenization, which splits words into subwords. We need to align labels properly.

In [ ]:
# Demonstrate subword tokenization
sample_tokens = ["Patient", "has", "hyperthyroidism", "."]
sample_labels = ["O", "O", "B-Disease", "O"]

print("Original tokens and labels:")
for token, label in zip(sample_tokens, sample_labels):
    print(f"  {token}: {label}")

# Tokenize with BioBERT tokenizer
tokenized = tokenizer(
    sample_tokens,
    is_split_into_words=True,
    return_tensors="pt",
    padding=True,
)

print("\nSubword tokens:")
subword_tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])
for i, token in enumerate(subword_tokens):
    print(f"  {i}: {token}")

# Word IDs show which original word each subword belongs to
word_ids = tokenized.word_ids(0)
print(f"\nWord IDs: {word_ids}")

In [ ]:
# Align labels with subwords
def align_labels_with_tokens(labels, word_ids, label_to_id):
    """
    Align word-level labels with subword tokens.
    - First subword of a word gets the original label
    - Subsequent subwords get -100 (ignored in loss calculation)
    - Special tokens ([CLS], [SEP], [PAD]) get -100
    """
    aligned_labels = []
    previous_word_idx = None
    
    for word_idx in word_ids:
        if word_idx is None:
            # Special token
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            # First subword of a new word - use the original label
            label = labels[word_idx]
            aligned_labels.append(label_to_id.get(label, 0))
        else:
            # Continuation subword - ignore in loss
            aligned_labels.append(-100)
        
        previous_word_idx = word_idx
    
    return aligned_labels

# Test alignment
aligned = align_labels_with_tokens(sample_labels, word_ids, LABEL_TO_ID)

print("Label alignment:")
print(f"{'Subword':<20} {'Word ID':<10} {'Aligned Label':<15}")
print("-" * 45)
for token, word_id, label in zip(subword_tokens, word_ids, aligned):
    label_str = ID_TO_LABEL.get(label, 'IGNORE') if label != -100 else 'IGNORE'
    print(f"{token:<20} {str(word_id):<10} {label} ({label_str})")

## Summary

Key takeaways:

1. **BioBERT** has 110M parameters with 768-dim hidden size
2. **BC5CDR** contains ~1500 abstracts with Disease and Chemical entities
3. **BIO tagging** uses B- (beginning), I- (inside), O (outside) labels
4. **Subword alignment** is critical - use word_ids() and set -100 for ignored tokens
5. Our label scheme has 7 labels: O, B/I-Disease, B/I-Chemical, B/I-Symptom

Next steps:
- Implement `src/dataset.py` with proper tokenization and alignment
- Implement `src/model.py` with LoRA configuration
- Train and evaluate!